In [4]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict 
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

response = llm.invoke("Hi")

print(response.content)

Hello! How can I help you today?


Persistence:  Ability to save and restore the state of a workflow with time.
Not only final state, we can save the state at each stage of the workflow

Checkpointers: They lie on the edges in the workflow, checkpoint will save the current state before going to each level

Threads: Threads uniquely represent a single workflow, different initial states can be saved under different thread ids

Features: Fault tolerance, Human in the loop, short term memory and Time Travel

In [ ]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str
def generate_joke(state: JokeState):

    prompt = f'generate a one-liner joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}
def generate_explanation(state: JokeState):

    prompt = f'write a two-liner explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [9]:

config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'cat'}, config=config1)

{'topic': 'cat',
 'joke': "Why don't cats play poker in the wild?\n\nToo many cheetahs.",
 'explanation': 'The joke puns on "cheetahs" sounding like "cheaters," implying wild cats would cheat at poker.'}

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'cat', 'joke': "Why don't cats play poker in the wild?\n\nToo many cheetahs.", 'explanation': 'The joke puns on "cheetahs" sounding like "cheaters," implying wild cats would cheat at poker.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac6bd-d9da-60c6-8002-37409d35eca9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-09T16:31:10.476908+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac6bd-cfae-6adb-8001-6867d00b2fee'}}, tasks=(), interrupts=())

In [11]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cat', 'joke': "Why don't cats play poker in the wild?\n\nToo many cheetahs.", 'explanation': 'The joke puns on "cheetahs" sounding like "cheaters," implying wild cats would cheat at poker.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac6bd-d9da-60c6-8002-37409d35eca9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-09T16:31:10.476908+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac6bd-cfae-6adb-8001-6867d00b2fee'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'cat', 'joke': "Why don't cats play poker in the wild?\n\nToo many cheetahs."}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac6bd-cfae-6adb-8001-6867d00b2fee'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-09-09T16:31:09.410561+00:00', parent_config={'

In [12]:

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pig'}, config=config2)

{'topic': 'pig',
 'joke': 'Why did the pig join a band?\n\nBecause he was tired of being a *boar*-ing solo act and wanted to try some *ham*-onies.',
 'explanation': 'The pig wanted to stop being boring and alone, seeking musical harmony with others instead.'}

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pig', 'joke': 'Why did the pig join a band?\n\nBecause he was tired of being a *boar*-ing solo act and wanted to try some *ham*-onies.', 'explanation': 'The pig wanted to stop being boring and alone, seeking musical harmony with others instead.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac6c2-956d-6ffe-8002-5215258e6850'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-09T16:33:17.520048+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac6c2-8fc6-6fac-8001-3178e4e6bd4f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pig', 'joke': 'Why did the pig join a band?\n\nBecause he was tired of being a *boar*-ing solo act and wanted to try some *ham*-onies.'}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac6c2-8fc6-6fac-8001-3178e4e6bd4f'}}, met

## LangGraph Persistence: Key Use Cases

### 1. Fault Tolerance
- The `checkpointer` saves the **entire graph state** after every node execution, tied to a `thread_id`.
- If the workflow crashes (exception, server restart, network failure) mid-execution, the state up to the last successfully completed node is **not lost** — it's safely stored in the checkpoint.
- To recover, simply call `workflow.invoke(None, config=config)` with the **same `thread_id`**. LangGraph automatically resumes from the last checkpoint instead of restarting from `START`.
- This avoids re-running already-completed (and possibly expensive/non-idempotent) steps like LLM calls or API calls.
- **Core idea:** you never lose more work than the single step that failed.

---

### 2. Time Travel (using `checkpoint_id`)
- Every node execution creates a **new checkpoint**, each with a unique `checkpoint_id`, forming a history of states over time for a given `thread_id`.
- `workflow.get_state_history(config)` lets you view all past checkpoints (like a version history / git log of your graph's execution).
- You can pass a specific `checkpoint_id` in the config to **"rewind"** the graph to that exact point in its execution:
```python
  config = {"configurable": {"thread_id": "1", "checkpoint_id": "<some_past_id>"}}
```
- From that rewound state, you can:
  - Simply inspect what the state looked like at that point (debugging).
  - Or **re-run/fork** the workflow from that checkpoint with modified input, creating an alternate execution branch — without affecting the original timeline.
- Useful for debugging, auditing decisions, and exploring "what-if" scenarios.

---

### 3. Human-in-the-Loop (HITL)
- Persistence enables **pausing** the graph at a specific node and waiting for human input before continuing.
- Achieved using `interrupt()` inside a node (or `interrupt_before` / `interrupt_after` when compiling the graph) — execution halts and the state is checkpointed right there.
- Because the state is saved, the graph doesn't need to stay "alive" in memory while waiting — a human can review/edit the state hours or days later, and the workflow will resume exactly where it left off.
- To resume:
```python
  workflow.invoke(Command(resume=<human_input>), config=config)
```
  LangGraph picks up from the interrupted node with the human's input incorporated.
- Common uses: approval steps (approve/reject an AI-drafted email), editing intermediate state (correcting an AI's extracted data before it proceeds), or injecting a decision the model can't make on its own.

---

**Common thread across all three:** persistence turns the *graph state* into something durable and addressable (`thread_id` + `checkpoint_id`), which is what makes recovery, rewinding, and pausing-for-humans all possible.

In [14]:
#TIME TRAVEL , you can go to a particular step in the workflow and re-run from there, this can be done using checkpoint id from the get_state_history method, you can also use the get_state method.

workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1ac6c2-871d-64a4-bfff-ffa1f9f02e17"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1ac6c2-871d-64a4-bfff-ffa1f9f02e17'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [16]:
workflow.invoke({'topic':'rat'}, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1ac6c2-871d-64a4-bfff-ffa1f9f02e17"}})

{'topic': 'rat',
 'joke': 'Why did the rat refuse to join the band?\n\nBecause he was already a master of the *squeak*-tacular solo.',
 'explanation': 'He was a "squeak-tacular" soloist—too good for a band, and he’s a rat, so squeaks.'}

### You can update the state values at any step using workflow.update_state() function

In [ ]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})